In [ ]:
# --- setup -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/pxr-repo'
!git clone -q https://github.com/pridem755/patient-or-xray.git $REPO 2>/dev/null || (cd $REPO && git pull -q)
%pip install -q -e $REPO

In [ ]:
import importlib
import site
import sys

site.main()
importlib.invalidate_caches()
SRC = f'{REPO}/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import torch

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable one in Runtime > Change runtime type')

In [ ]:
from pathlib import Path

import pandas as pd

from pxr.config import load_config
from pxr.data.cache import load_cache
from pxr.data.splits import assemble_out_of_fold, fold_membership
from pxr.model.train import predict, train_fold, training_config_from

cfg = load_config(f'{REPO}/config/study_config.yaml')
ROOT = Path(cfg.paths['drive_root'])
COHORTS = ROOT / cfg.paths['cohorts']
SPLITS = ROOT / cfg.paths['splits']
CACHE = ROOT / cfg.paths['image_cache']
MODELS = ROOT / cfg.paths['models']
SCORES = ROOT / cfg.paths['scores']
MODELS.mkdir(parents=True, exist_ok=True)
SCORES.mkdir(parents=True, exist_ok=True)

train_cfg = training_config_from(cfg)
print('config_hash :', cfg.config_hash)
print('architecture :', train_cfg.architecture, f'({train_cfg.image_size}px)')
print('labels :', len(train_cfg.labels))
print('batch / lr :', train_cfg.batch_size, '/', train_cfg.lr)
print('epochs :', train_cfg.max_epochs, f'(patience {train_cfg.patience},'
      f' warmup {train_cfg.warmup_epochs})')
print('mixed precision:', train_cfg.mixed_precision)
print('pos weighting :', train_cfg.positive_weighting, f'(cap {train_cfg.max_positive_weight})')
print('augmentation : rotation ±{}°, translate {}, scale {} — no horizontal flip'
      .format(train_cfg.rotation_degrees, train_cfg.translate_fraction, train_cfg.scale_range))

In [ ]:
# --- load cohorts and caches ------------------------------------------------------
cohorts, caches = {}, {}
folds = pd.read_parquet(SPLITS / f'folds_{cfg.split_hash}.parquet')

for site in cfg.training_sites:
    cohort = pd.read_parquet(COHORTS / cfg.artifact_name('cohort', site=site))
    cohort = cohort.merge(folds[folds.site == site][['patient_id', 'fold']],
                          on='patient_id', how='left')
    assert cohort['fold'].notna().all(), f'{site}: some patients have no fold'
    cohorts[site] = cohort
    caches[site] = load_cache(CACHE / site, cohort_ids=cohort['image_id'],
                              config_hash=cfg.cohort_hash)
    print(f'{site:<10} {len(cohort):>7,} patients, cache verified')

In [ ]:
# --- visualize some cached pixels -----------------------------------------------
import matplotlib.pyplot as plt

from pxr.data.cache import read_images

site = cfg.training_sites[0]
sample = cohorts[site].sample(6, random_state=0)
pixels = read_images(caches[site], sample['image_id'])

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, (_, row), img in zip(axes, sample.iterrows(), pixels):
    ax.imshow(img, cmap='gray')
    positive = [l for l in cfg.analysis_labels if row.get(l) == 1]
    ax.set_title(f"{row['view']} {row['sex'][0]}{int(row['age'])}\n"
                 f"{', '.join(positive[:2]) or 'none'}", fontsize=8)
    ax.axis('off')
plt.suptitle(f'{site}: cached pixels beside their cohort labels')
plt.tight_layout()
plt.show()

In [ ]:
# --- check fold membership ------------------------------------------------------
def roles_for(site, fold):
    return fold_membership(
        fold, cohorts[site],
        stratify_by=cfg.stratify_by,
        val_stratify_by=cfg.val_stratify_by,
        n_folds=cfg.n_folds,
        val_fraction=cfg.val_fraction,
        min_val_stratum=cfg.min_val_stratum,
        seed=cfg.splits['seed'],
    )

role = roles_for(cfg.training_sites[0], 0)
print(role.value_counts(normalize=True).round(3).to_dict())
print(f'train {int((role == "train").sum()):,} | val {int((role == "val").sum()):,} '
      f'| test {int((role == "test").sum()):,}')

In [ ]:
# --- train a single fold --------------------------------------------------------
SITE, FOLD = cfg.training_sites[0], 0

role = roles_for(SITE, FOLD)
train_frame = cohorts[SITE][role == 'train']
val_frame = cohorts[SITE][role == 'val']
print(f'{SITE} fold {FOLD}: train {len(train_frame):,}  val {len(val_frame):,}\n')

model, history = train_fold(
    train_frame, val_frame, caches[SITE], train_cfg,
    site=SITE, fold=FOLD, checkpoint_dir=MODELS,
)
print('\n' + history.summary())

In [ ]:
frame = history.to_frame()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(frame.epoch, frame.train_loss, label='train')
axes[0].plot(frame.epoch, frame.val_loss, label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend()
axes[1].plot(frame.epoch, frame.val_macro_auroc, marker='o')
axes[1].axhline(0.85, ls='--', c='grey', lw=1)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val macro-AUROC')
axes[1].set_title('dashed line: published-baseline range')
plt.tight_layout(); plt.show()

In [ ]:
# --- train all folds -----------------------------------------------------------
import torch

histories = []
for site in cfg.training_sites:
    for fold in range(cfg.n_folds):
        checkpoint = MODELS / f'{site}_fold{fold}.pt'
        if checkpoint.exists():
            print(f'{site} fold {fold}: checkpoint exists, skipping')
            continue

        print(f'\n=== {site} fold {fold} ===')
        role = roles_for(site, fold)
        model, history = train_fold(
            cohorts[site][role == 'train'],
            cohorts[site][role == 'val'],
            caches[site], train_cfg,
            site=site, fold=fold, checkpoint_dir=MODELS,
        )
        histories.append(history)
        print(history.summary())
        del model
        torch.cuda.empty_cache()

In [ ]:
# --- generate out-of-fold predictions ------------------------------------------
from pxr.model.train import build_model

for site in cfg.training_sites:
    out_path = SCORES / f'oof_{site}_{cfg.model_hash}.parquet'
    if out_path.exists():
        print(f'{site}: predictions exist, skipping')
        continue

    per_fold = {}
    for fold in range(cfg.n_folds):
        checkpoint = MODELS / f'{site}_fold{fold}.pt'
        if not checkpoint.exists():
            raise FileNotFoundError(f'{checkpoint} missing — train fold {fold} first')

        model = build_model(train_cfg.architecture, len(train_cfg.labels), pretrained=False)
        model.load_state_dict(torch.load(checkpoint, map_location='cpu'))

        role = roles_for(site, fold)
        per_fold[fold] = predict(model, cohorts[site][role == 'test'],
                                 caches[site], train_cfg)
        print(f'  {site} fold {fold}: {len(per_fold[fold]):,} predictions')
        del model
        torch.cuda.empty_cache()

    pooled = assemble_out_of_fold(per_fold, cohorts[site], n_folds=cfg.n_folds)
    pooled.to_parquet(out_path, index=False)
    print(f'{site}: {len(pooled):,} out-of-fold predictions -> {out_path.name}\n')

In [ ]:
# --- evaluate out-of-fold predictions ------------------------------------------
from pxr.model.train import macro_auroc

rows = []
for site in cfg.training_sites:
    pooled = pd.read_parquet(SCORES / f'oof_{site}_{cfg.model_hash}.parquet')
    merged = cohorts[site][['patient_id', *cfg.analysis_labels]].merge(
        pooled, on='patient_id', suffixes=('_true', '_pred'))
    truth = merged[[f'{l}_true' for l in cfg.analysis_labels]].to_numpy(dtype=float)
    score = merged[[f'{l}_pred' for l in cfg.analysis_labels]].to_numpy(dtype=float)
    macro, per_label = macro_auroc(truth, score)
    rows.append({'site': site, 'macro_auroc': round(macro, 4),
                 **{l: round(v, 4) for l, v in zip(cfg.analysis_labels, per_label)}})

performance = pd.DataFrame(rows)
print(performance.to_string(index=False))

In [ ]:
# --- integrity cell------------------
print(f'config_hash : {cfg.config_hash}')
print(f'architecture: {train_cfg.architecture}, {train_cfg.image_size}px, '
      f'batch {train_cfg.batch_size}, lr {train_cfg.lr}')
print(f'folds : {cfg.n_folds} per site, sites {cfg.training_sites}')
for site in cfg.training_sites:
    pooled = pd.read_parquet(SCORES / f'oof_{site}_{cfg.model_hash}.parquet')
    print(f'{site:<10} {len(pooled):>7,} out-of-fold predictions')